In [ ]:
## import dependencies
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
import pandas as pd
import torch
import plotly
import requests
#import STalign
#from STalign import STalign
import pickle
import json
import cv2
import scanpy as sc
import anndata as ad
import muon
from scipy import sparse
from torch import nn

# make plots bigger
plt.rcParams["figure.figsize"] = (12,10)

In [ ]:
### import adata frist
he_id = 'S17_19647_02_05'
adata = sc.read_h5ad(f'{he_id}_RNA_adata.h5ad')

In [ ]:
adata

In [ ]:
adata.X = adata.layers['counts'].copy()

In [ ]:
#sc.pp.calculate_qc_metrics(adata, inplace=True, log1p=True)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
#sc.pp.highly_variable_genes(adata, n_top_genes=2000)

In [ ]:
# scale data, clip values exceeding standard deviation 10.
sc.pp.scale(adata, max_value=10)

sc.tl.pca(adata, svd_solver='arpack')
# plot more PCS
sc.pl.pca(adata, components = ['1,2','3,4','5,6','7,8'], ncols=2)

In [ ]:
sc.pl.pca_variance_ratio(adata, log=True, n_pcs = 50)

In [ ]:
# Then we calculate a new umap
sc.pp.neighbors(adata, n_neighbors=100, n_pcs=25, use_rep='X_pca', metric='cosine', random_state=0)

sc.tl.umap(adata, min_dist=0.0, random_state=0)

In [ ]:
sc.tl.leiden(adata, resolution = 0.1, key_added = "leiden_0.1", random_state=0, flavor='igraph', n_iterations=2)
sc.tl.leiden(adata, resolution = 0.15, key_added = "leiden_0.15", random_state=0, flavor='igraph', n_iterations=2)
sc.tl.leiden(adata, resolution = 0.2, key_added = "leiden_0.2", random_state=0, flavor='igraph', n_iterations=2)

In [ ]:
adata_protein = sc.read_h5ad(f'{he_id}_protein_adata.h5ad')

In [ ]:
adata_protein

In [ ]:
protein_names = [protein[:-16] for protein in adata_protein.var_names]
adata_protein.var_names = protein_names

In [ ]:
sc.pl.spatial(adata_protein, img_key=None, color='leiden_0.6', size=4)

In [ ]:
sc.tl.rank_genes_groups(adata_protein, 'leiden_0.6', method='wilcoxon', key_added = "wilcoxon_0.6")
top_genes = pd.DataFrame(adata_protein.uns['wilcoxon_0.6']['names']).head(10)
top_genes

In [ ]:
adata_protein.var_names_make_unique()

In [ ]:
# 1. Define the clusters we want to look at (0-11 from Out[57])
# 2. Extract top markers for each cluster to use in the dotplot
# We will take the top 3 markers per cluster for clarity
adata_protein.var_names_make_unique()
result = adata_protein.uns['wilcoxon_0.6']
groups = result['names'].dtype.names
markers_to_plot = {}

for group in groups:
    # Get top 3 markers for this cluster
    top_markers = list(result['names'][group][1:7])
    markers_to_plot[group] = top_markers

# 3. Generate the DotPlot
sc.pl.dotplot(
    adata_protein, 
    markers_to_plot, 
    groupby='leiden_0.6', 
    standard_scale='var',  # Scales expression 0 to 1 for better visualization
    #lowest_fraction=0.1,   # Don't show dots if <10% of cells express it
    colorbar_title='Mean expression\n(scaled)',
    title='Top Protein Markers per Cluster (Res 0.6)'
)

Based on the protein marker results in **Out[57]** of your [STAlign-citeseq-RNA_liver-0205-annotation notebook](https://ood-mccleary.ycrc.yale.edu/node/r814u03n02.mccleary.ycrc.yale.edu/62113/notebooks/ycga_work/STAlign/STAlign-citeseq-RNA_liver-0205-annotation.ipynb), here are the specific protein markers for Clusters 5 and 8 that indicate metabolic and fibrosis-related stress:

### Cluster 5: Metabolic Stress Markers

Cluster 5 is characterized by proteins associated with acute metabolic dysfunction and altered lipid signaling:

* **CD142 (Tissue Factor):** Ranked 5th. In the liver, CD142 is often upregulated in response to inflammatory metabolic stress and is a key initiator of the coagulation cascade, which is frequently activated in MASH.
* **LOX-1 (Lectin-like oxidized LDL receptor 1):** Ranked 1st. This is a primary scavenger receptor for oxidized LDL. High expression in these cells indicates a high level of oxidative stress and lipid-induced injury (lipotoxicity).
* **TIGIT:** Ranked 6th. While often considered an immune checkpoint, its presence in this cluster suggests a specialized stressed state or interaction with the inflammatory microenvironment.

https://pmc.ncbi.nlm.nih.gov/articles/PMC8545876/

https://www.ahajournals.org/doi/10.1161/ATVBAHA.120.315421

### Cluster 8: Fibrosis-Related Stress Markers

Cluster 8 markers highlight the transition from metabolic injury to the physical remodeling of the liver tissue:

* **SLAM (Signaling Lymphocytic Activation Molecule):** Ranked 1st. SLAM family members are involved in the crosstalk between hepatocytes and immune cells during the progression of chronic liver injury and fibrosis.
* **TREM-1 (Triggering Receptor Expressed on Myeloid cells 1):** Ranked 5th. In the context of MASH, TREM-1 is a major driver of the "fibro-inflammatory" response, amplifying the signals that lead to stellate cell activation.
* **Integrin α7:** Ranked 6th. Integrins are structural proteins that mediate cell-matrix interactions. The upregulation of α7 in this cluster suggests hepatocytes are physically responding to the stiffening of the extracellular matrix (the developing scar tissue).
* **CLEC12A:** Ranked 3rd (and 8th). This is a sensor for cell death and damage-associated molecular patterns (DAMPs), indicating these cells are in an area of high turnover and active tissue damage.

https://www.ncbi.nlm.nih.gov/books/NBK6259/

### Summary for MASH Context:

* **Cluster 5** represents cells in the **Lipotoxic Phase**, dealing with the immediate chemical stress of oxidized lipids (LOX-1).
* **Cluster 8** represents cells in the **Fibrotic Phase**, expressing the machinery needed to interact with the scarring environment (Integrins) and amplify inflammatory damage (TREM-1).

In [ ]:
adata.obs['leiden_0.4_protein'] = adata_protein.obs['leiden_0.4'].copy()
adata.obs['leiden_0.6_protein'] = adata_protein.obs['leiden_0.6'].copy()

In [ ]:
adata.uns['leiden_0.6_protein_colors'] = ['#1f77b4',
 '#ff7f0e',
 '#279e68',
 '#d62728',
 '#aa40fc',
 '#FFFF00',#'#8c564b',
 '#e377c2',
 '#b5bd61',
 '#17becf',
 '#aec7e8',
 '#ffbb78',
 '#98df8a']

In [ ]:
adata.X = adata.layers['counts'].copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden_0.6_protein', method='wilcoxon', key_added = "wilcoxon_0.6_protein")
top_genes = pd.DataFrame(adata.uns['wilcoxon_0.6_protein']['names']).head(20)
top_genes

In [ ]:
import pandas as pd

# 1. Extract the log-fold changes from the results stored in uns
# The key 'wilcoxon_0.6_protein' was created in your previous Cell [19]
lfc_results = pd.DataFrame(adata.uns['wilcoxon_0.6_protein']['logfoldchanges'])

# 2. Extract the gene names to use as a corresponding reference
names_results = pd.DataFrame(adata.uns['wilcoxon_0.6_protein']['names'])

# 3. Optional: Create a combined view for a specific cluster (e.g., Cluster 11)
# This creates a table with Gene Name and its corresponding Log Fold Change
cluster_id = '11'
cluster_11_fc = pd.DataFrame({
    'gene': names_results[cluster_id],
    'logfoldchange': lfc_results[cluster_id]
}).head(20)

print(f"Top markers and log-fold changes for Cluster {cluster_id}:")
cluster_11_fc

In [ ]:
# 1. Define the mapping from cluster numbers to new cell type names
cluster_map = {
    "0": "P0: Endothelial Cells (LSECs)",
    "1": "P1: Hepatocytes \n(Quiescent)",
    "2": "P2: Hepatocytes \n(Periportal)",
    "3": "P3: MASH-Associated Macrophages", # \n(MAMs)
    "4": "P4: Ductular Reaction \n(Biliary-like)",
    "5": "P5: Metabolic Stress \nHepatocytes",
    "6": "P6: Mature Cholangiocytes",
    "7": "P7: Portal Progenitor-like Cells",
    "8": "P8: Fibrosis",
    "9": "P9: Activated Hepatic Stellate Cells", #\n(aHSCs)
    "10": "P10: Myofibroblasts (Active Fibrosis)",
    "11": "P11: Fibro-Inflammatory Progenitors"
}

# 2. Map the existing categories to the new names
# This updates the 'leiden_0.6_protein' column in place
adata.obs['leiden_0.6_protein_ann'] = (
    adata.obs['leiden_0.6_protein']
    .map(cluster_map)
    .astype('category')
)

adata.uns['leiden_0.6_protein_ann_colors'] = adata.uns['leiden_0.6_protein_colors'].copy()


In [ ]:
# 3. Verify the renaming by plotting the spatial distribution with new names
import matplotlib.pyplot as plt

# Define the plotting context for a dark theme, high resolution, and large font
with plt.rc_context({
    'figure.dpi': 300, 
    'axes.facecolor': 'black', 
    'figure.facecolor': 'black',
    'text.color': 'white',
    'axes.labelcolor': 'white',
    'xtick.color': 'white',
    'ytick.color': 'white',
    'font.size': 20,           # Sets global font size to 20
    'legend.facecolor': 'black',
    'legend.edgecolor': 'white'
}):
    # Generate the spatial plot
    ax = sc.pl.spatial(
        adata, 
        color='leiden_0.6_protein_ann', 
        size=4, 
        title='',#"Annotated Liver Clusters (MASH)", 
        img_key=None, 
        frameon=False,
        show=False
    )
    
    # Customize the legend specifically for white text and size 20
    legend = plt.legend(
        bbox_to_anchor=(0.92, 0.9), 
        loc='upper left', 
        frameon=False,
        fontsize=19,          # Explicitly set legend font size
        title_fontsize=15,     # Set legend title size if applicable
        markerscale=2.0
    )
    
    # Explicitly ensure all legend text is white
    for text in legend.get_texts():
        text.set_color("white")
        
    plt.show()

In [ ]:
differ_genes = ['DLC1', 'FP671120.4', 'MALAT1', 'FNBP1', 'ADGRL3', 'CELF2', 'MCTP1', 'RNF8', 'TRIM14', 'CYB5D2', 'TMEM212', 
                'C11orf58', 'WRN', 'HPR', 'COTL1', 'FOXB1', 'NTAQ1', 'AVEN', 'LPP', 'ADAMTS16']

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

# list of genes defined in your notebook (Cell 128)
# differ_genes = ['Grp', 'Tmem108', 'Chrd', ...] 
adata_gene_threshold = adata.copy()
percentile = 25

for gene in differ_genes:
    if gene in adata_gene_threshold.var_names:
        # Extract expression values for the specific gene
        # We use .toarray() in case the matrix is sparse
        gene_values = adata_gene_threshold[:, gene].layers['counts'].flatten()
        
        # Calculate the 5th percentile threshold
        threshold = np.percentile(gene_values, percentile)
        
        # Find indices of cells below this threshold
        low_indices = adata_gene_threshold[:, gene].layers['counts'] < threshold
        
        # Set those values to zero in the main matrix
        # Note: If using sparse matrices, this approach is efficient
        adata_gene_threshold.layers['counts'][low_indices.flatten(), adata_gene_threshold.var_names == gene] = 0

print(f"Lower {percentile}% of cells for each gene in differ_genes have been set to zero.")


#plt.rcParams.update({'font.size': 20, 'axes.titlesize': 14, 'axes.labelsize': 14})
# It is recommended to use a subset or specific markers for readability 
# as differ_genes contains 48 genes.
dp=sc.pl.dotplot(
    adata_gene_threshold, 
    layer='counts',
    var_names=differ_genes, 
    groupby='leiden_0.6_protein_ann', 
    standard_scale='var',  # Scales expression 0 to 1 per gene for better visualization
    #cmap='viridis',
    title='Expression of marker genes across cell types', 
    figsize=(40, 15),
        show=False
)

# 2. Access the main plot axis
ax = dp['mainplot_ax']

# 3. MANUALLY RESCALE DOTS: 
# Since figsize is large, we tell the path collection to increase the base size
from matplotlib.collections import PathCollection
for child in ax.get_children():
    if isinstance(child, PathCollection):
        # Multiply the existing sizes by a factor (e.g., 2.5) 
        # to match the 15x10 figure size
        child.set_sizes(child.get_sizes() * 8)

# 4. Styling (Remove frame and increase font)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='both', labelsize=30)
plt.xticks(fontsize=25)
plt.yticks(fontsize=25)

# 5. INCREASE LEGEND SIZE
# Access the legend axes and update the font sizes
# 'size_legend_ax' controls the dot size legend
# 'color_legend_ax' controls the expression colorbar legend
if 'size_legend_ax' in dp:
    dp['size_legend_ax'].tick_params(labelsize=25)
    dp['size_legend_ax'].set_title('Fraction of cells\nin group (%)', fontsize=25)

if 'color_legend_ax' in dp:
    dp['color_legend_ax'].tick_params(labelsize=25)
    dp['color_legend_ax'].set_ylabel('Mean expression\nin group', fontsize=25)

plt.show()

In [ ]:
# 1. Generate the DotPlot with a large figsize
dp = sc.pl.dotplot(
    adata_protein_sub, 
    stress_protein_markers, 
    groupby='celltype', 
    standard_scale='var', 
    figsize=(15, 10),  # Large figure size
    dot_max=1.0,       # Use the full range for the largest dots
    dot_min=0.1,
    show=False
)

# 2. Access the main plot axis
ax = dp['mainplot_ax']

# 3. MANUALLY RESCALE DOTS: 
# Since figsize is large, we tell the path collection to increase the base size
from matplotlib.collections import PathCollection
for child in ax.get_children():
    if isinstance(child, PathCollection):
        # Multiply the existing sizes by a factor (e.g., 2.5) 
        # to match the 15x10 figure size
        child.set_sizes(child.get_sizes() * 15)

# 4. Styling (Remove frame and increase font)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='both', labelsize=45)
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)

plt.show()

In [ ]:
THRB_hepatocytes = adata[:, 'THRB'].X.toarray().flatten() * ind_hepatocytes
ACACA_hepatocytes = adata[:, 'ACACA'].X.toarray().flatten() * ind_hepatocytes
adata.obs['THRB in hepatocytes'] = THRB_hepatocytes
adata.obs['ACACA in hepatocytes'] = ACACA_hepatocytes

In [ ]:
with plt.rc_context({
    'figure.dpi': 300, 
    'axes.facecolor': 'white', 
    'figure.facecolor': 'black',
    'text.color': 'white',
    'axes.labelcolor': 'white',
    'xtick.color': 'white',
    'ytick.color': 'white',
    'font.size': 20,           # Sets global font size to 20
    'legend.facecolor': 'white',
    'legend.edgecolor': 'white'
}):
    sc.pl.spatial(adata, color=['ACACA in hepatocytes'], size=4.5, img_key=None, frameon=False, vmin=0.0132, vmax=0.018)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Set global figure parameters for high resolution
sc.set_figure_params(figsize=(16, 8), fontsize=20, dpi=300)

# 2. Generate the plot (using ACACA as an example from your Cell 163)
ax = sc.pl.violin(
    adata[adata_protein_sub.obs_names, :], 
    keys='ACACA', 
    groupby='leiden_0.6_protein_ann', 
    show=False
)

# 3. Remove top and right frame (spines)
sns.despine(ax=ax, top=True, right=True)

# 4. Remove guidelines (grid)
ax.grid(False)

# 5. Increase font size for x and y axis ticks and labels
ax.tick_params(axis='both', which='major', labelsize=45)
ax.set_xlabel('', fontsize=35)
ax.set_ylabel(ax.get_ylabel(), fontsize=55)

# 6. Final adjustments
plt.ylim([0.008, 0.03])
#plt.xticks(ha='right')
plt.show()